# 任务 3：黄斑中央凹定位 · Kaggle 完整版

**目标：** 使用 80 张有标签图像，预测 20 张测试图的中央凹原图像素坐标，严格按模板生成 40 行 `submission.csv`。

**当前交付状态：代码已完成语法和合成数据逻辑检查，尚未在你的 Kaggle 数据或 GPU 上实际训练。不能预先保证 RMSE 或比赛排名。**

### 运行方法
1. 在 Kaggle Notebook 中导入本文件，添加 `2026task3` 比赛数据。
2. 开启 **GPU T4 / P100**。首次使用 ImageNet 权重需开启 Internet；无网络时可将官方 ResNet34 权重作为数据添加，再填写 `LOCAL_PRETRAINED`。
3. 路径已按你提供的位置填写。按顺序运行全部代码格；第一格主要定义函数，最后一格启动完整流程。
4. 运行完成后，下载并提交 `/kaggle/working/submission.csv`。不需要手动创建表头、拼接 X/Y 或调整行顺序。

### 方法与预算
- ImageNet 预训练 ResNet34 + 多尺度特征融合 + 位置热图；可识别的黄斑框用于辅助区域监督。
- 保持长宽比缩放至 384×384，保留并反变换像素坐标；增强时图像、点和框同步变换。
- 5 折交叉验证，每折最多 60 轮；按验证 RMSE 保存最佳权重，支持早停。
- 原图与水平翻转 TTA，平均 5 折预测后输出坐标。
- **累计训练预算 95 分钟**，本次运行软预算 110 分钟，并预留预测和导出时间。训练较快时会提前完成，不会为了用满预算重复训练。
- 预算检查按 batch/epoch 执行。GPU 性能、预训练权重下载、磁盘 IO 会影响实际耗时，因此不是严格的两小时执行保证。

默认只使用一张 GPU，无需安装额外深度学习包。默认使用预训练权重；若比赛 Rules 禁止，先把 `CFG['pretrained']` 改为 `False`，同时预期小样本效果可能下降。

## 1 · 路径、参数与数据检查函数
通常直接运行，不用修改。模板在比赛目录中自动查找；多个模板或特殊 ImageID 命名无法确认时，会在开始训练前报出具体问题。

`train_location` 支持 Pascal VOC XML 和标准 YOLO 五列 txt。只采用与真实坐标一致且可唯一识别的框；未支持的格式会跳过并报告。坐标 CSV 是主要监督来源。

坐标默认按原图零起点像素坐标解释，图像不会自动进行 EXIF 旋转。运行时会显示真实标注预览；如果红色十字明显不在对应位置，请停止并核对标注。

In [ ]:
"""Kaggle task 3: fovea localization, 5-fold heatmaps, bounded training.
Run this file in Kaggle with a CUDA GPU. No training has been run by the author.
"""
# %% Configuration and data utilities
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import os
import random
import re
import socket
import time
import warnings
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights
from sklearn.model_selection import KFold, StratifiedKFold, StratifiedGroupKFold

ROOT = Path('/kaggle/input/competitions/2026task3/detection')
TRAIN_DIR = ROOT / 'train'
TEST_DIR = ROOT / 'test'
BOX_DIR = ROOT / 'train_location'
GT_PATH = ROOT / 'fovea_localization_train_GT.csv'
SAMPLE_PATH = ''                 # 自动查找；找到多个模板时再填写绝对路径
OUT = Path('/kaggle/working/fovea_task3_v1')
RESUME_SOURCE = ''               # 新会话恢复时，填已添加的旧输出目录；平时留空
LOCAL_PRETRAINED = ''            # 无网络时，填官方 ResNet34 权重 .pth 的路径
SUBMISSION_ID_MAP = {}           # 特殊模板可手填：{'模板ID': ('图片.jpg', 'x')}

CFG = dict(
    version='fovea_heatmap_v1.1', seed=2026,
    size=384, batch_size=8, folds=5, epochs=60,
    encoder_lr=1e-4, head_lr=8e-4, weight_decay=1e-4,
    min_epochs=18, patience=12, sigma=2.0,
    pretrained=True, use_boxes=True, workers=0,
    train_budget_minutes=95, total_budget_minutes=110,
    visibility_threshold=0.5,
)
# 95 分钟是各折累计训练预算；110 分钟为本次运行的软预算，预留导出时间。
# 每个 batch 检查时间，但下载、文件 IO、CUDA 调用无法被严格定时中断。
# Rules 若禁止外部预训练权重，必须将 pretrained 改为 False。
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig', dtype=str, keep_default_na=False)


def file_digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def image_index(folder):
    files = sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    if not files:
        raise FileNotFoundError(f'没有图片：{folder}')
    by_name, by_stem = defaultdict(list), defaultdict(list)
    for p in files:
        by_name[p.name.lower()].append(p)
        by_stem[p.stem.lower()].append(p)
    if any(len(v) > 1 for v in by_name.values()):
        raise ValueError(f'{folder} 存在同名图片，需先明确标注对应关系。')
    return files, by_name, by_stem


def resolve_image(value, index):
    name = str(value).strip().replace('\\', '/').split('/')[-1]
    _, by_name, by_stem = index
    matches = by_name.get(name.lower(), []) or by_stem.get(Path(name).stem.lower(), [])
    if len(matches) != 1:
        raise ValueError(f'无法唯一匹配图像：{value!r}；候选={matches}')
    return matches[0]


def normalize_id(value):
    return re.sub(r'[^a-z0-9\u4e00-\u9fff]', '', str(value).lower())


def map_template(sample, test_files, overrides=None):
    """Resolve actual IDs before training; never invent row order or naming."""
    columns = {str(c).strip().lower(): c for c in sample.columns}
    if set(columns) != {'imageid', 'value'} or len(sample.columns) != 2:
        raise ValueError(f'模板应只有 ImageID、value 两列，实际：{list(sample.columns)}')
    id_col, value_col = columns['imageid'], columns['value']
    if len(sample) != 2 * len(test_files) or sample[id_col].duplicated().any():
        raise ValueError(f'模板必须每图两行且 ID 不重复；图片 {len(test_files)}，行数 {len(sample)}')
    aliases = defaultdict(set)
    names = {p.name for p in test_files}
    for p in test_files:
        for base in {p.name, p.stem}:
            for axis in ('x', 'y'):
                for text in (f'{base}_{axis}', f'{base}_Fovea_{axis}',
                             f'{axis}_{base}', f'Fovea_{axis}_{base}',
                             f'{p.stem}_{axis}{p.suffix}', f'{p.stem}_Fovea_{axis}{p.suffix}'):
                    aliases[normalize_id(text)].add((p.name, axis))
    mapping, missing = [], []
    for value in sample[id_col]:
        if overrides and str(value) in overrides:
            name, axis = overrides[str(value)]
            candidates = {(name, str(axis).lower())}
        else:
            candidates = aliases.get(normalize_id(value), set())
        if len(candidates) != 1:
            missing.append(str(value))
        else:
            item = next(iter(candidates))
            if item[0] not in names or item[1] not in ('x', 'y'):
                raise ValueError(f'手工 ID 映射无效：{value} -> {item}')
            mapping.append(item)
    if missing:
        raise ValueError('模板 ID 不能安全匹配，训练尚未开始。\n'
                         f'未匹配 ID：{missing[:10]}\n'
                         f'测试图片：{[p.name for p in test_files[:10]]}\n'
                         '请把这些输出发给我；不要自行改模板行顺序。')
    expected = {(p.name, a) for p in test_files for a in ('x', 'y')}
    if len(set(mapping)) != len(mapping) or set(mapping) != expected:
        raise ValueError('模板映射没有恰好覆盖每张图片的 X、Y。')
    return id_col, value_col, mapping


def find_template(test_files):
    if SAMPLE_PATH:
        candidates = [Path(SAMPLE_PATH)]
    else:
        # 包含比赛根目录，适配模板与 detection 目录并列的情况。
        candidates = sorted(p for p in ROOT.parent.rglob('*.csv')
                            if p.name.lower() in {'sample_submission.csv', 'submission.csv'})
    good, errors = [], []
    for p in candidates:
        try:
            sample = read_csv(p)
            mapping = map_template(sample, test_files, SUBMISSION_ID_MAP)
            good.append((p, sample, mapping))
        except Exception as exc:
            errors.append(f'{p}: {exc}')
    if len(good) != 1:
        if len(good) > 1:
            raise ValueError(f'有多个可用模板，请设置 SAMPLE_PATH：{[str(t[0]) for t in good]}')
        raise ValueError('未找到可安全使用的提交模板。请设置 SAMPLE_PATH。\n' + '\n'.join(errors[:3]))
    return good[0]


def letterbox_meta(width, height, size):
    scale = min(size / width, size / height)
    nw, nh = max(1, round(width * scale)), max(1, round(height * scale))
    return dict(w=width, h=height, sx=nw / width, sy=nh / height,
                left=(size - nw) // 2, top=(size - nh) // 2, nw=nw, nh=nh)


def original_to_unit(xy, meta, size):
    # 像素中心约定使翻转 x -> 1-x 和反变换严格一致。
    return np.array([((xy[0] + .5) * meta['sx'] + meta['left']) / size,
                     ((xy[1] + .5) * meta['sy'] + meta['top']) / size], dtype=np.float32)


def unit_to_original(xy, metas, size):
    xy = np.asarray(xy, dtype=np.float64)
    result = np.empty_like(xy)
    for i, m in enumerate(metas):
        result[i, 0] = np.clip((xy[i, 0] * size - m['left']) / m['sx'] - .5, 0, m['w'] - 1)
        result[i, 1] = np.clip((xy[i, 1] * size - m['top']) / m['sy'] - .5, 0, m['h'] - 1)
    return result


def load_letterboxed(path, meta, size):
    with Image.open(path) as image:
        # 不自动应用 EXIF 旋转，标注按原文件的像素布局处理。
        rgb = np.asarray(image.convert('RGB'))
    small = cv2.resize(rgb, (meta['nw'], meta['nh']), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    canvas[meta['top']:meta['top'] + meta['nh'], meta['left']:meta['left'] + meta['nw']] = small
    return canvas


def parse_box_file(path, width, height):
    """Only explicit Pascal VOC XML / normalized YOLO detection txt formats."""
    boxes = []
    if path.suffix.lower() == '.xml':
        root = ET.parse(path).getroot()
        aw = float(root.findtext('size/width') or width)
        ah = float(root.findtext('size/height') or height)
        if aw <= 0 or ah <= 0:
            raise ValueError('XML 图像尺寸无效')
        for obj in root.findall('.//object'):
            b = obj.find('bndbox')
            if b is None:
                continue
            coords = [float(b.findtext(k)) for k in ('xmin', 'ymin', 'xmax', 'ymax')]
            coords = np.array(coords) * np.array([width / aw, height / ah] * 2)
            boxes.append((obj.findtext('name') or '', coords))
    elif path.suffix.lower() == '.txt':
        for line in path.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            values = line.split()
            if len(values) != 5:
                raise ValueError('不是标准的 YOLO class cx cy w h 五列格式')
            label, cx, cy, bw, bh = map(float, values)
            if label < 0 or not label.is_integer() or not all(0 <= v <= 1 for v in (cx, cy, bw, bh)):
                raise ValueError('YOLO 类别或归一化坐标不合法')
            boxes.append((str(int(label)), np.array([(cx-bw/2)*width, (cy-bh/2)*height,
                                                    (cx+bw/2)*width, (cy+bh/2)*height])))
    return boxes


def choose_box(candidates, xy, width, height):
    valid = []
    for label, b in candidates:
        b = np.asarray(b, dtype=float)
        if not np.isfinite(b).all():
            continue
        b[[0, 2]] = b[[0, 2]].clip(0, width)
        b[[1, 3]] = b[[1, 3]].clip(0, height)
        if b[0] <= xy[0] <= b[2] and b[1] <= xy[1] <= b[3] and b[2] > b[0] and b[3] > b[1]:
            valid.append((label, b))
    named = [b for label, b in valid if any(k in label.lower() for k in ('fovea', 'macula', '黄斑', '中央凹'))]
    pool = named or [b for _, b in valid]
    # 多个不同框时不猜；同一框同时有 XML/txt 副本则去重。
    unique = {tuple(np.round(b, 2)): b for b in pool}
    return next(iter(unique.values())).tolist() if len(unique) == 1 else None


def prepare_data():
    for p in (TRAIN_DIR, TEST_DIR, GT_PATH):
        if not p.exists():
            raise FileNotFoundError(f'路径不存在：{p}。请确认比赛数据已添加到 Notebook。')
    train_index, test_index = image_index(TRAIN_DIR), image_index(TEST_DIR)
    test_files = test_index[0]
    gt = read_csv(GT_PATH)
    gt.columns = [str(c).strip() for c in gt.columns]
    required = ['data', 'Fovea_X', 'Fovea_Y']
    if not all(c in gt.columns for c in required):
        raise ValueError(f'坐标 CSV 列名不匹配：{list(gt.columns)}')
    print('坐标标注前五行：\n', gt.head().to_string(index=False))
    print('train_location 文件类型：', Counter(p.suffix.lower() for p in BOX_DIR.rglob('*') if p.is_file()))
    ann_index = defaultdict(list)
    if CFG['use_boxes'] and BOX_DIR.exists():
        for p in BOX_DIR.rglob('*'):
            if p.suffix.lower() in {'.xml', '.txt'}:
                key = p.stem.lower()
                if Path(key).suffix.lower() in IMAGE_EXTS:
                    key = Path(key).stem
                ann_index[key].append(p)
    train, tests, cache = [], [], {}
    errors, seen = [], set()
    for _, row in gt.iterrows():
        p = resolve_image(row['data'], train_index)
        if p in seen:
            raise ValueError(f'同一训练图片被重复标注：{p.name}')
        seen.add(p)
        xy = np.array([float(row['Fovea_X']), float(row['Fovea_Y'])])
        with Image.open(p) as image:
            width, height = image.size
        if not np.isfinite(xy).all() or (xy < 0).any() or xy[0] >= width or xy[1] >= height:
            raise ValueError(f'{p.name} 坐标 {xy} 超出原图 {width}x{height}，请核对坐标约定。')
        visible = int(not np.all(xy == 0))
        meta = letterbox_meta(width, height, CFG['size'])
        candidates = []
        for ann in ann_index.get(p.stem.lower(), []):
            try:
                candidates.extend(parse_box_file(ann, width, height))
            except Exception as exc:
                errors.append(f'{ann.name}: {exc}')
        box = choose_box(candidates, xy, width, height) if visible else None
        record = dict(name=p.name, path=str(p), xy=xy.tolist(), visible=visible,
                      meta=meta, box=box, group=file_digest(p))
        train.append(record)
        cache[str(p)] = load_letterboxed(p, meta, CFG['size'])
    if len(seen) != len(train_index[0]):
        raise ValueError(f'训练图 {len(train_index[0])} 张，CSV 只覆盖 {len(seen)} 张，请先核对数据。')
    for p in test_files:
        with Image.open(p) as image:
            width, height = image.size
        meta = letterbox_meta(width, height, CFG['size'])
        tests.append(dict(name=p.name, path=str(p), meta=meta, xy=[0, 0], visible=0, box=None))
        cache[str(p)] = load_letterboxed(p, meta, CFG['size'])
    template_path, sample, mapping = find_template(test_files)
    counts = Counter(r['visible'] for r in train)
    if counts[1] < 2:
        raise ValueError('可见中央凹样本太少，无法训练和验证定位模型。')
    print(f'训练 {len(train)} 张 | 测试 {len(tests)} 张 | 可见 {counts[1]} | 不可见 {counts[0]}')
    if len(train) != 80 or len(tests) != 20:
        warnings.warn('图片数量与题干 80/20 不一致，请检查输出；按实际数据继续。')
    print('可安全使用的辅助标注框：', sum(r['box'] is not None for r in train))
    if errors:
        print('以下标注未使用（前 5 条）：', *errors[:5], sep='\n')
    if not any(r['box'] is not None for r in train):
        print('本轮仅使用坐标监督。未把未知格式的框猜成黄斑标签。')
    if counts[0] == 0:
        warnings.warn('训练集中没有 (0,0)：无法训练不可见判断，本轮全部按可见处理；请检查测试预览。')
    elif counts[0] < CFG['folds']:
        warnings.warn('不可见样本少于折数，部分验证折无法充分验证不可见判断。')
    print('提交模板：', template_path)
    print(sample.head(6).to_string(index=False))
    print('前六行实际映射：', mapping[2][:6])
    return train, tests, cache, sample, mapping


def make_splits(records):
    n = len(records)
    y = np.array([r['visible'] for r in records])
    groups = np.array([r['group'] for r in records])
    if len(set(groups)) < CFG['folds']:
        raise ValueError('独立图片数量少于折数。')
    if len(set(groups)) < n:
        print('检测到完全相同的图像，采用分组划分防止重复图泄漏。')
        splitter = StratifiedGroupKFold(CFG['folds'], shuffle=True, random_state=CFG['seed'])
        splits = list(splitter.split(np.zeros(n), y, groups))
    elif len(set(y)) == 2 and min(Counter(y).values()) >= CFG['folds']:
        splits = list(StratifiedKFold(CFG['folds'], shuffle=True, random_state=CFG['seed']).split(np.zeros(n), y))
    else:
        splits = list(KFold(CFG['folds'], shuffle=True, random_state=CFG['seed']).split(np.zeros(n)))
    if any(not y[tr].any() for tr, _ in splits):
        raise ValueError('某折训练集没有可见样本，请减少折数。')
    return splits


class FoveaDataset(Dataset):
    def __init__(self, records, cache, augment=False):
        self.records, self.cache, self.augment = records, cache, augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        r = self.records[index]
        size = CFG['size']
        image = self.cache[r['path']].copy()
        xy = original_to_unit(r['xy'], r['meta'], size) if r['visible'] else np.array([.5, .5], np.float32)
        mask = np.zeros((size, size), dtype=np.float32)
        has_box = float(r['box'] is not None)
        if has_box:
            x1, y1, x2, y2 = r['box']
            m = r['meta']
            x1, x2 = [int(np.clip(round(x*m['sx'] + m['left']), 0, size)) for x in (x1, x2)]
            y1, y2 = [int(np.clip(round(y*m['sy'] + m['top']), 0, size)) for y in (y1, y2)]
            mask[y1:y2, x1:x2] = 1.
        if self.augment:
            M = cv2.getRotationMatrix2D(((size-1)/2, (size-1)/2), random.uniform(-15, 15), random.uniform(.92, 1.08))
            M[:, 2] += np.random.uniform(-.04, .04, 2) * size
            candidate = (M @ np.r_[xy*size-.5, 1.] + .5) / size
            if not r['visible'] or np.all((candidate > .015) & (candidate < .985)):
                image = cv2.warpAffine(image, M, (size, size), flags=cv2.INTER_LINEAR, borderValue=0)
                mask = cv2.warpAffine(mask, M, (size, size), flags=cv2.INTER_NEAREST, borderValue=0)
                xy = candidate.astype(np.float32)
            if random.random() < .5:
                image, mask = image[:, ::-1].copy(), mask[:, ::-1].copy()
                xy[0] = 1. - xy[0]
            gamma = random.uniform(.85, 1.18)
            lut = np.clip((np.arange(256)/255.)**gamma * 255., 0, 255).astype(np.uint8)
            image = cv2.LUT(image, lut).astype(np.float32)
            image = image * random.uniform(.85, 1.15) + random.uniform(-8, 8)
            image *= np.random.uniform(.95, 1.05, (1, 1, 3))
            image = np.clip(image, 0, 255).astype(np.uint8)
            if random.random() < .12:
                image = cv2.GaussianBlur(image, (3, 3), .6)
        x = torch.from_numpy(image.transpose(2, 0, 1).copy()).float().div_(255.)
        x = (x - torch.tensor([.485, .456, .406])[:, None, None]) / torch.tensor([.229, .224, .225])[:, None, None]
        return x, torch.tensor(xy, dtype=torch.float32), torch.tensor(float(r['visible'])), torch.from_numpy(mask[None]), torch.tensor(has_box)


def loader(records, cache, augment=False):
    return DataLoader(FoveaDataset(records, cache, augment), batch_size=CFG['batch_size'],
                      shuffle=augment, num_workers=CFG['workers'], pin_memory=True, drop_last=False)

## 2 · 模型与训练函数
验证集不做随机增强。比赛 RMSE 的计算口径为：

$$\mathrm{RMSE}=\sqrt{\frac{1}{2N}\sum_{i=1}^{N}[(\hat x_i-x_i)^2+(\hat y_i-y_i)^2]}.$$

包含不可见样本，并在原图像素单位下计算；没有把欧氏距离平均值当成 RMSE。

可见性阈值固定为 0.5，不反复按 80 张图搜索阈值。若某折训练集没有 `(0,0)`，该折将全部按可见处理；如果全体训练集没有不可见样本，就无法从这些标签中学习可靠的不可见判别。

In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, channels=96):
        super().__init__(nn.Conv2d(channels, channels, 3, padding=1, bias=False),
                         nn.GroupNorm(8, channels), nn.SiLU(inplace=True))


class FoveaNet(nn.Module):
    def __init__(self, initialize=False):
        super().__init__()
        self.encoder = resnet34(weights=None)
        if initialize and CFG['pretrained']:
            if LOCAL_PRETRAINED:
                state = torch_load(LOCAL_PRETRAINED, weights_only=True)
            else:
                old_timeout = socket.getdefaulttimeout()
                socket.setdefaulttimeout(60)
                try:
                    state = ResNet34_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)
                except Exception as exc:
                    raise RuntimeError('ImageNet 权重下载失败。开启 Kaggle Internet，或把官方 resnet34-b627a593.pth 添加为数据并填写 LOCAL_PRETRAINED。未静默改为随机初始化。') from exc
                finally:
                    socket.setdefaulttimeout(old_timeout)
            self.encoder.load_state_dict(state, strict=True)
        self.encoder.fc = nn.Identity()
        self.lateral = nn.ModuleList([nn.Conv2d(c, 96, 1) for c in (64, 128, 256, 512)])
        self.smooth = nn.ModuleList([ConvBlock() for _ in range(3)])
        self.heatmap = nn.Sequential(ConvBlock(), nn.Conv2d(96, 1, 1))
        self.region = nn.Conv2d(96, 1, 1)
        self.visible = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(.2), nn.Linear(128, 1))

    def train(self, mode=True):
        super().train(mode)
        if mode and CFG['pretrained']:
            # 小样本时保留预训练 BN 的统计量，避免每批统计量剧烈波动。
            for m in self.encoder.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        return self

    def forward(self, x):
        e = self.encoder
        x = e.maxpool(e.relu(e.bn1(e.conv1(x))))
        c1 = e.layer1(x)
        c2 = e.layer2(c1)
        c3 = e.layer3(c2)
        c4 = e.layer4(c3)
        features = [c1, c2, c3, c4]
        p = self.lateral[3](c4)
        for i in (2, 1, 0):
            p = self.smooth[i](F.interpolate(p, size=features[i].shape[-2:], mode='bilinear', align_corners=False) + self.lateral[i](features[i]))
        return self.heatmap(p), self.visible(F.adaptive_avg_pool2d(c4, 1).flatten(1)).squeeze(1), self.region(p)


def heatmap_xy(prob, local=False):
    b, _, height, width = prob.shape
    xx = (torch.arange(width, device=prob.device, dtype=torch.float32) + .5) / width
    yy = (torch.arange(height, device=prob.device, dtype=torch.float32) + .5) / height
    p = prob.float()
    if local:
        peak = p.flatten(1).argmax(1)
        px, py = peak % width, peak // width
        gx = torch.arange(width, device=p.device)[None, None, None, :]
        gy = torch.arange(height, device=p.device)[None, None, :, None]
        keep = ((gx - px[:, None, None, None]).abs() <= 4) & ((gy - py[:, None, None, None]).abs() <= 4)
        p = p * keep
        p = p / p.sum((1, 2, 3), keepdim=True).clamp_min(1e-12)
    return torch.stack([(p * xx[None, None, None, :]).sum((1, 2, 3)),
                        (p * yy[None, None, :, None]).sum((1, 2, 3))], dim=1)


def training_loss(outputs, xy, visible, mask, has_box, train_visibility):
    logits, vlogits, region = outputs
    logits, vlogits, region = logits.float(), vlogits.float(), region.float()
    b, _, h, w = logits.shape
    gx = (torch.arange(w, device=logits.device).float() + .5)[None, None, None, :]
    gy = (torch.arange(h, device=logits.device).float() + .5)[None, None, :, None]
    target = torch.exp(-((gx-xy[:, 0, None, None, None]*w)**2 + (gy-xy[:, 1, None, None, None]*h)**2) / (2*CFG['sigma']**2))
    target = target / target.sum((1, 2, 3), keepdim=True).clamp_min(1e-12)
    logp = logits.flatten(1).log_softmax(1)
    hm_loss = -(target.flatten(1) * logp).sum(1)
    probs = logp.exp().reshape_as(logits)
    coord_loss = (heatmap_xy(probs) - xy).square().mean(1)
    loss = ((hm_loss + 30*coord_loss) * visible).sum() / visible.sum().clamp_min(1)
    if train_visibility:
        loss = loss + .5 * F.binary_cross_entropy_with_logits(vlogits, visible)
    if has_box.sum() > 0:
        target_mask = F.interpolate(mask, size=(h, w), mode='area')
        bce = F.binary_cross_entropy_with_logits(region, target_mask, reduction='none').mean((1, 2, 3))
        pred_mask = region.sigmoid()
        dice = 1-(2*(pred_mask*target_mask).sum((1, 2, 3))+1)/((pred_mask+target_mask).sum((1, 2, 3))+1)
        loss = loss + .15 * ((bce+dice)*has_box).sum()/has_box.sum()
    return loss


def amp_context():
    return torch.autocast(device_type='cuda', dtype=torch.float16)


def make_scaler():
    try:
        return torch.amp.GradScaler('cuda')
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()


def torch_load(path, weights_only=False):
    try:
        return torch.load(path, map_location='cpu', weights_only=weights_only)
    except TypeError:
        return torch.load(path, map_location='cpu')


def atomic_save(obj, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    torch.save(obj, temp)
    os.replace(temp, path)


def write_json(obj, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temp, path)


def competition_rmse(pred, truth):
    return float(np.sqrt(np.mean((np.asarray(pred, np.float64)-np.asarray(truth, np.float64))**2)))


@torch.inference_mode()
def predict(model, records, cache, force_visible, tta=False):
    model.eval()
    xy_all, visible_all = [], []
    for x, *_ in loader(records, cache):
        x = x.cuda(non_blocking=True)
        with amp_context():
            heat, v, _ = model(x)
        p = heat.float().flatten(1).softmax(1).reshape_as(heat)
        vp = v.float().sigmoid()
        if tta:
            with amp_context():
                hf, vf, _ = model(x.flip(-1))
            pf = hf.float().flatten(1).softmax(1).reshape_as(hf).flip(-1)
            p = (p + pf) * .5
            vp = (vp + vf.float().sigmoid()) * .5
        xy_all.append(heatmap_xy(p, local=True).cpu().numpy())
        visible_all.append(np.ones(len(x), np.float32) if force_visible else vp.cpu().numpy())
    xy = unit_to_original(np.concatenate(xy_all), [r['meta'] for r in records], CFG['size'])
    return xy, np.concatenate(visible_all)


def apply_visibility(xy, p):
    result = np.asarray(xy).copy()
    result[np.asarray(p) < CFG['visibility_threshold']] = 0.
    return result


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all())


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])


def train_fold(fold, tr, va, cache, fingerprint, progress, wall_deadline):
    folder = OUT / f'fold_{fold}'
    folder.mkdir(parents=True, exist_ok=True)
    best_path, last_path = folder/'best.pt', folder/'last.pt'
    info = progress['folds'][fold]
    if info.get('done') and best_path.exists():
        print(f'Fold {fold}: 复用已完成模型。')
        return best_path
    remaining = CFG['train_budget_minutes']*60 - sum(s['seconds'] for s in progress['folds'])
    unfinished = sum(not s.get('done', False) for s in progress['folds'])
    allowance = min(remaining, max(0, wall_deadline-time.monotonic())) / max(unfinished, 1)
    if allowance < 30:
        if best_path.exists():
            info['done'] = True
            write_json(progress, OUT/'progress.json')
            return best_path
        raise RuntimeError('剩余时间不足以训练所有折。已有模型已保存；检查 GPU 或增加预算后重新运行。')
    start_time = time.monotonic()
    deadline = start_time + allowance
    seed_all(CFG['seed'] + fold)
    model = FoveaNet(initialize=not last_path.exists()).cuda()
    enc_ids = {id(p) for p in model.encoder.parameters()}
    head = [p for p in model.parameters() if id(p) not in enc_ids]
    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': CFG['encoder_lr'], 'initial_lr': CFG['encoder_lr']},
        {'params': head, 'lr': CFG['head_lr'], 'initial_lr': CFG['head_lr']},
    ], weight_decay=CFG['weight_decay'])
    scaler = make_scaler()
    start_epoch, best, bad = 0, float('inf'), 0
    force_visible = all(r['visible'] for r in tr)
    if last_path.exists():
        ckpt = torch_load(last_path)
        if ckpt['fingerprint'] != fingerprint:
            raise RuntimeError('断点与当前数据/参数不同。请改 OUT 为新目录后运行。')
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch, best, bad = ckpt['epoch'] + 1, ckpt['best'], ckpt['bad']
        restore_rng(ckpt['rng'])
        print(f'Fold {fold}: 从第 {start_epoch+1} 轮继续，已有最佳 RMSE={best:.2f}')
        del ckpt
    dl = loader(tr, cache, augment=True)
    truth = np.array([r['xy'] for r in va], dtype=float)
    consumed_before = info['seconds']
    history_path = folder/'history.csv'
    try:
        for epoch in range(start_epoch, CFG['epochs']):
            if time.monotonic() >= deadline and best_path.exists():
                break
            epoch_start = time.monotonic()
            warm = min(1., (epoch+1)/3)
            cosine = .05 + .95*(1+math.cos(math.pi*max(0, epoch-3)/max(1, CFG['epochs']-3)))/2
            for group in optimizer.param_groups:
                group['lr'] = group['initial_lr'] * warm * cosine
            model.train()
            total_loss, count = 0., 0
            for x, xy, visible, mask, has_box in dl:
                if time.monotonic() >= deadline and count > 0:
                    break
                x, xy, visible, mask, has_box = [v.cuda(non_blocking=True) for v in (x, xy, visible, mask, has_box)]
                optimizer.zero_grad(set_to_none=True)
                with amp_context():
                    outputs = model(x)
                # 损失在 FP32 中计算，避免 heatmap 的 softmax 下溢。
                loss = training_loss(outputs, xy, visible, mask, has_box, not force_visible)
                if not torch.isfinite(loss):
                    raise RuntimeError('损失出现 NaN/Inf，训练已停止；请发送当前日志。')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.)
                scaler.step(optimizer)
                scaler.update()
                total_loss += float(loss.detach()) * len(x)
                count += len(x)
            raw, probability = predict(model, va, cache, force_visible, tta=False)
            score = competition_rmse(apply_visibility(raw, probability), truth)
            if score < best:
                best, bad = score, 0
                atomic_save(dict(model=model.state_dict(), fingerprint=fingerprint,
                                 epoch=epoch, rmse=score, force_visible=force_visible), best_path)
            else:
                bad += 1
            atomic_save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                             scaler=scaler.state_dict(), epoch=epoch, best=best, bad=bad,
                             fingerprint=fingerprint, rng=rng_state()), last_path)
            seconds = time.monotonic()-epoch_start
            log = pd.DataFrame([dict(epoch=epoch+1, loss=total_loss/max(1,count),
                                     val_rmse=score, best_rmse=best, seconds=seconds)])
            log.to_csv(history_path, mode='a', header=not history_path.exists(), index=False)
            info['seconds'] = consumed_before + time.monotonic()-start_time
            info['epoch'] = epoch+1
            info['best_rmse'] = best
            write_json(progress, OUT/'progress.json')
            print(f'fold {fold+1}/{CFG["folds"]} | epoch {epoch+1:02d} | '
                  f'loss {total_loss/max(1,count):.4f} | val RMSE {score:.2f} | '
                  f'best {best:.2f} | {seconds:.1f}s', flush=True)
            if epoch == start_epoch:
                print(f'本折剩余预算约 {max(0,(deadline-time.monotonic())/60):.1f} 分钟；按实测速度动态截停。')
            if bad >= CFG['patience'] and epoch+1 >= CFG['min_epochs']:
                print('本折早停。')
                break
        if not best_path.exists():
            raise RuntimeError('本折尚未完成一次训练验证，未生成可用模型。')
        if info.get('epoch', 0) < CFG['min_epochs']:
            warnings.warn(f'Fold {fold} 因时间预算较短，只完成 {info.get("epoch", 0)} 轮，请结合验证结果判断是否欠拟合。')
        info['done'] = True
    finally:
        info['seconds'] = consumed_before + time.monotonic()-start_time
        write_json(progress, OUT/'progress.json')
        del model, optimizer, scaler
        torch.cuda.empty_cache()
    return best_path

## 3 · 交叉验证、可视化和导出函数
- `oof_rmse`：每张训练图由未用它训练的折模型预测，再统一计算 RMSE；各折仍用自己的验证集选择最佳轮数，因此不是额外独立测试分数。
- `baseline_rmse`：各折只用其训练集估计的平均位置基线，用于检查模型是否确实优于简单猜测。
- `visible_rmse`：只统计真实可见图；主指标仍是包含全部图像的 `oof_rmse`。
- 图像级交叉验证自动将完全相同的文件分在同一组。若数据有同一患者的多张不同图，还应提供患者编号以改成患者级分组验证。

测试坐标融合先平均可见性概率和原始位置，再做一次 `(0,0)` 判定，避免将不同折输出的零坐标直接混进位置平均。

In [ ]:
def fingerprint_for(records, splits):
    keys = ['version', 'seed', 'size', 'batch_size', 'folds', 'epochs', 'encoder_lr',
            'head_lr', 'weight_decay', 'min_epochs', 'patience', 'sigma', 'pretrained',
            'use_boxes', 'visibility_threshold']
    payload = dict(config={k: CFG[k] for k in keys},
                   pretrained_digest=file_digest(Path(LOCAL_PRETRAINED)) if LOCAL_PRETRAINED else 'torchvision_resnet34_v1',
                   data=[{k: r[k] for k in ('name', 'xy', 'visible', 'group', 'box', 'meta')} for r in records],
                   splits=[(a.tolist(), b.tolist()) for a, b in splits])
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def export_submission(sample, mapping, test_records, predictions, destination):
    id_col, value_col, pairs = mapping
    pred = np.asarray(predictions, dtype=np.float64)
    if pred.shape != (len(test_records), 2) or not np.isfinite(pred).all():
        raise ValueError('预测形状不正确或存在 NaN/Inf。')
    names = [r['name'] for r in test_records]
    if len(names) != len(set(names)):
        raise ValueError('测试图片同名，不能安全导出。')
    by_name = dict(zip(names, pred))
    result = sample.copy()
    result[value_col] = [float(by_name[name][0 if axis == 'x' else 1]) for name, axis in pairs]
    assert list(result.columns) == list(sample.columns)
    assert result[id_col].tolist() == sample[id_col].tolist()
    assert len(result) == 2*len(test_records)
    result.to_csv(destination, index=False)
    return result


def preview(records, pred, path, truth=None, probabilities=None):
    import matplotlib.pyplot as plt
    n = len(records)
    fig, axes = plt.subplots(math.ceil(n/4), 4, figsize=(16, 4*math.ceil(n/4)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for i, (ax, r) in enumerate(zip(axes.flat, records)):
        with Image.open(r['path']) as image:
            image = image.convert('RGB')
            w, h = image.size
            image.thumbnail((640, 640))
            ax.imshow(image, extent=(-.5, w-.5, h-.5, -.5))
        if np.any(pred[i] != 0):
            ax.scatter(*pred[i], c='red', marker='+', s=130, linewidths=2, label='prediction')
        if truth is not None and np.any(truth[i] != 0):
            ax.scatter(*truth[i], facecolors='none', edgecolors='lime', s=130, linewidths=2, label='GT')
        title = r['name']
        if truth is not None:
            title += f'\nEuclidean error {np.linalg.norm(pred[i]-truth[i]):.1f}px'
        elif probabilities is not None:
            title += f'\np_visible={probabilities[i]:.2f}'
        if np.all(pred[i] == 0):
            title += ' | predicted (0,0)'
        ax.set_title(title, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)


def restore_outputs():
    if RESUME_SOURCE:
        import shutil
        source = Path(RESUME_SOURCE)
        if not (source/'progress.json').exists():
            raise FileNotFoundError('RESUME_SOURCE 必须指向含 progress.json 和 fold_* 的旧输出目录。')
        if OUT.exists() and any(OUT.iterdir()):
            print('当前 OUT 已有输出，优先使用当前目录，不覆盖。')
        else:
            shutil.copytree(source, OUT, dirs_exist_ok=True)
            print('已恢复旧输出；将校验指纹后复用。')
    OUT.mkdir(parents=True, exist_ok=True)


def main():
    run_start = time.monotonic()
    if not torch.cuda.is_available():
        raise RuntimeError('没有可用 GPU。请在 Kaggle Notebook Settings 中开启 GPU T4/P100 后重跑。CPU 不适合本次两小时预算。')
    if CFG['size'] % 32 or CFG['folds'] < 2:
        raise ValueError('size 须为 32 的倍数，folds 至少为 2。')
    print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
    print('仅使用 cuda:0；T4 x2 环境中第二张卡不参与本方案。')
    seed_all(CFG['seed'])
    cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark = True
    restore_outputs()
    train, tests, cache, sample, mapping = prepare_data()
    # 训练前显示一组真实标注，可及时发现 CSV 坐标约定或图片匹配错误。
    labelled = [r for r in train if r['visible']][:12]
    preview(labelled, np.array([r['xy'] for r in labelled]), OUT/'training_labels.png')
    print('训练标注预览：红色 + 为 CSV 中的真实坐标，请留意是否落在黄斑中央凹附近。')
    try:
        from IPython.display import display, Image as DisplayImage
        display(DisplayImage(filename=str(OUT/'training_labels.png')))
    except ImportError:
        pass
    splits = make_splits(train)
    fingerprint = fingerprint_for(train, splits)
    progress_path = OUT/'progress.json'
    if progress_path.exists():
        progress = json.loads(progress_path.read_text())
        if progress['fingerprint'] != fingerprint:
            raise RuntimeError('当前数据或训练配置与旧输出不同，请把 OUT 改为新目录（如 fovea_task3_v2）。')
    else:
        progress = dict(fingerprint=fingerprint, folds=[dict(seconds=0., done=False) for _ in splits])
        write_json(progress, progress_path)
    write_json(CFG, OUT/'config.json')
    split_rows = []
    for f, (_, va) in enumerate(splits):
        split_rows.extend(dict(data=train[int(i)]['name'], fold=f) for i in va)
    pd.DataFrame(split_rows).to_csv(OUT/'folds.csv', index=False)
    wall_deadline = run_start + (CFG['total_budget_minutes']-10)*60
    models = []
    for f, (tr, va) in enumerate(splits):
        models.append(train_fold(f, [train[int(i)] for i in tr], [train[int(i)] for i in va],
                                 cache, fingerprint, progress, wall_deadline))
    print('训练阶段结束，开始原图/水平翻转 TTA 与五折融合。')
    oof_raw = np.full((len(train), 2), np.nan)
    oof_p = np.full(len(train), np.nan)
    baseline = np.full((len(train), 2), np.nan)
    test_xy, test_p = [], []
    for f, ((tr, va), checkpoint_path) in enumerate(zip(splits, models)):
        ckpt = torch_load(checkpoint_path)
        if ckpt['fingerprint'] != fingerprint:
            raise RuntimeError('模型指纹不一致。')
        model = FoveaNet(initialize=False).cuda()
        model.load_state_dict(ckpt['model'])
        val_records = [train[int(i)] for i in va]
        raw, probability = predict(model, val_records, cache, ckpt['force_visible'], tta=True)
        oof_raw[va], oof_p[va] = raw, probability
        raw_test, probability_test = predict(model, tests, cache, ckpt['force_visible'], tta=True)
        test_xy.append(raw_test)
        test_p.append(probability_test)
        # 简单参照：仅由本折训练集估计可见位置均值，未使用验证坐标。
        visible_train = [train[int(i)] for i in tr if train[int(i)]['visible']]
        norm_mean = np.mean([np.array(r['xy'])/[r['meta']['w'], r['meta']['h']] for r in visible_train], axis=0)
        baseline[va] = np.array([norm_mean*[r['meta']['w'], r['meta']['h']] for r in val_records])
        if np.mean([train[int(i)]['visible'] for i in tr]) < CFG['visibility_threshold']:
            baseline[va] = 0
        del model, ckpt
        torch.cuda.empty_cache()
    truth = np.array([r['xy'] for r in train], dtype=float)
    visibility = np.array([r['visible'] for r in train], dtype=bool)
    oof = apply_visibility(oof_raw, oof_p)
    assert np.isfinite(oof).all()
    # 先平均各折可见性概率和坐标，再做一次不可见判定；不把零坐标混入位置平均。
    ensemble_p = np.mean(test_p, axis=0)
    ensemble_raw = np.mean(test_xy, axis=0)
    ensemble = apply_visibility(ensemble_raw, ensemble_p)
    disagreement = np.sqrt(np.sum(np.std(test_xy, axis=0)**2, axis=1))
    metrics = dict(oof_rmse=competition_rmse(oof, truth),
                   baseline_rmse=competition_rmse(baseline, truth),
                   visible_rmse=competition_rmse(oof[visibility], truth[visibility]),
                   visibility_accuracy=float(np.mean((oof_p >= CFG['visibility_threshold']) == visibility)),
                   training_minutes=sum(s['seconds'] for s in progress['folds'])/60,
                   note='OOF 使用各折验证集选择最佳轮数，并非额外独立测试；未按公开榜调参。')
    write_json(metrics, OUT/'metrics.json')
    print('\n验证结果（单位：原图像素；RMSE 越低越好）：')
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    if metrics['oof_rmse'] >= metrics['baseline_rmse']:
        warnings.warn('模型未优于均值位置基线，请先检查验证误差图与训练日志，不要据此声称有效提升。')
    pd.DataFrame(dict(data=[r['name'] for r in train], true_x=truth[:,0], true_y=truth[:,1],
                      pred_x=oof[:,0], pred_y=oof[:,1], p_visible=oof_p,
                      euclidean_error=np.linalg.norm(oof-truth, axis=1))).to_csv(OUT/'oof_predictions.csv', index=False)
    coords = pd.DataFrame(dict(data=[r['name'] for r in tests], Fovea_X=ensemble[:,0],
                               Fovea_Y=ensemble[:,1], p_visible=ensemble_p,
                               fold_disagreement_px=disagreement))
    coords.to_csv(OUT/'test_coordinates.csv', index=False)
    submission = export_submission(sample, mapping, tests, ensemble, OUT/'submission.csv')
    # 同时放到 working 根目录，便于下载和直接提交。
    submission.to_csv('/kaggle/working/submission.csv', index=False)
    worst = np.argsort(np.linalg.norm(oof-truth, axis=1))[-12:][::-1]
    preview([train[int(i)] for i in worst], oof[worst], OUT/'validation_errors.png', truth=truth[worst])
    preview(tests, ensemble, OUT/'test_predictions.png', probabilities=ensemble_p)
    print('\n提交文件检查通过：', len(submission), '行，列名', list(submission.columns))
    print(submission.head(8).to_string(index=False))
    print('\n本次运行耗时：', round((time.monotonic()-run_start)/60, 1), '分钟')
    print('提交文件：/kaggle/working/submission.csv')
    print('请查看测试预览；下载/保存整个 fovea_task3_v1 输出，避免会话回收后丢失模型。')
    try:
        from IPython.display import display, Image as DisplayImage
        display(DisplayImage(filename=str(OUT/'validation_errors.png')))
        display(DisplayImage(filename=str(OUT/'test_predictions.png')))
    except ImportError:
        pass
    return metrics

## 4 · 开始运行
这一格会执行：数据检查 → 标注预览 → 5 折训练 → OOF 验证 → 测试预测 → 提交文件检查与保存。

输出会逐轮更新。若显存不足，把 `batch_size` 改为 4，并换一个新的 `OUT` 目录。不要在改模型参数后继续加载旧目录的模型。

**中断恢复：** 在保留 `/kaggle/working/fovea_task3_v1` 的同一会话，重跑本格即可。会复用已完成折，并从未完成折最后保存的轮次继续，训练时间累计计入预算。

**新会话恢复：** 必须先把此前保存的模型输出添加为 Kaggle 输入数据，再将 `RESUME_SOURCE` 指向包含 `progress.json` 和 `fold_*` 的目录。仅仅重新打开同一个 Notebook，并不代表模型文件和变量仍存在。

In [ ]:
metrics = main()

## 运行后的文件
| 文件 | 用途 |
| --- | --- |
| `/kaggle/working/submission.csv` | **比赛提交文件**，预期 40 行、ImageID/value 两列 |
| `fovea_task3_v1/metrics.json` | 原图像素 RMSE、基线、训练耗时 |
| `fovea_task3_v1/oof_predictions.csv` | 每张训练图的真实值、OOF 预测和误差 |
| `fovea_task3_v1/test_coordinates.csv` | 每张测试图的 X/Y、可见概率、各折分歧 |
| `fovea_task3_v1/training_labels.png` | 真实坐标预览；红色十字表示 CSV 标注 |
| `fovea_task3_v1/validation_errors.png` | 最差 12 张验证图；绿圈是真实位置，红十字是预测位置 |
| `fovea_task3_v1/test_predictions.png` | 20 张测试图的最终预测预览 |
| `fovea_task3_v1/fold_*/best.pt` | 每折最佳模型，用于预测 |
| `fovea_task3_v1/fold_*/last.pt` | 每折最近训练断点，含优化器和随机状态 |

请保存 Notebook 版本及 Output，或下载完整输出目录。`best.pt` 和 `last.pt` 较大，是保留训练成果所必需的文件。

运行后优先检查：实际读取图片数、不可见样本数、框使用数量、`oof_rmse` 是否低于 `baseline_rmse`，以及测试预测图。如果模板匹配失败，直接提供错误中的 ID 示例和图片名称即可，不用重复训练。

### 实现参考
- [TorchVision ResNet34 官方文档及 ImageNet 权重](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html)
- [PyTorch 自动混合精度训练官方示例](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)
- [Integral Human Pose Regression](https://arxiv.org/abs/1711.08229)：热图与坐标回归的相关方法参考；本代码采用自定义热图损失和局部概率加权坐标解码，并非该论文的完整复现。

本地已核验：600 组坐标缩放/翻转反变换，常见提交 ID 映射及错误拦截，40 行提交顺序，RMSE 分母与不可见样本处理，VOC/YOLO 框解析，五折覆盖与完全重复图像分组。**尚未验证模型在你这份数据上的实际分数及 GPU 执行。**